In [ ]:
from pathlib import Path
import pandas as pd

import teehr
from teehr.evaluation.spark_session_utils import create_spark_session
from teehr import RemoteReadWriteEvaluation
from teehr.utilities.apply_migrations import evolve_catalog_schema

#### Start the spark session

In [ ]:
spark = create_spark_session(
    aws_access_key_id="minioadmin",
    aws_secret_access_key="minioadmin123",
    update_configs={
        "spark.hadoop.fs.s3a.aws.credentials.provider":
        "org.apache.hadoop.fs.s3a.AnonymousAWSCredentialsProvider"
    }
)

#### Set up the "remote" data warehouse schema in your local cluster

In [ ]:
migrations_dir = Path(teehr.__file__).parent / "migrations"

evolve_catalog_schema(
    spark=spark,
    migrations_dir_path=migrations_dir,
    target_catalog_name="iceberg",
    target_namespace_name="teehr"
)

#### Initialize a TEEHR Evaluation with read/write privileges

In [ ]:
ev = RemoteReadWriteEvaluation(spark=spark)

#### Now you can pull in a subset of data from the TEEHR-Cloud warehouse via the API
Note: If you encounter a time out error, try reducing the page_size arg

In [ ]:
# Handpicked sites that seemed interesting
usgs_gages = [
    "usgs-02424000",
    "usgs-03068800",
    "usgs-01570500",
    "usgs-01347000",
    "usgs-05443500",
    "usgs-06770500",
    "usgs-08313000",
    "usgs-11421000",
    "usgs-14319500"
]

In [ ]:
%%time
ev.download.evaluation_subset(
    location_ids=usgs_gages,
    start_date="2019-01-01",
    end_date="2020-12-31",
    primary_configuration_name="usgs_observations",
    secondary_configuration_name="nwm30_retrospective",
    page_size=7500
)

To run the MAP workflow you need to add the basin pixel weights table manually at this time

In [ ]:
df = pd.read_parquet("local_data/dev_grid_pixel_coverage_weights.parquet")
ev.table("grid_pixel_coverage_weights").load_dataframe(
    df=df,
    write_mode="create_or_replace"
)

In [ ]:
ev.download.locations(
    ids=df.location_id.tolist(),
    load=True
)

In [ ]:
spark.stop()